# LeetCode 2886 — Change Data Type bằng Python (pandas)

Link đề: https://leetcode.com/problems/change-data-type/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def changeDatatype(students: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `students` có cột `grade` kiểu **số thực** (`float64`, ví dụ `73.0`); **đổi cột `grade` sang số nguyên** (`int64`, ví dụ `73`).
2. Hiểu khái niệm **kiểu dữ liệu (`dtype`)** của một cột và cách xem `df.dtypes`.
3. Đi từ cách **vòng lặp tự gọi `int(...)` từng ô** → **`astype(int)`** → **`astype({"grade": "int64"})` theo từ điển** → **`apply(int)` / `assign`** → **`pd.to_numeric(..., downcast=...)` và kiểu nullable `Int64`**.
4. Biết `astype(int)` **cắt bỏ phần thập phân** (không làm tròn), và **không chuyển được khi có giá trị thiếu** (`NaN`).
5. Tránh các lỗi kinh điển: quên gán lại kết quả `astype`, nhầm cắt với làm tròn, chuyển cột có `NaN`, chuyển chuỗi không phải số, và `return` sai kiểu.

## 2. Đề bài & ví dụ

Có một bảng `students` với các cột `student_id`, `name`, `age`, `grade`. Cột `grade` đang là **số thực** (`float64`) nhưng điểm thực chất là số nguyên.

Hãy viết hàm **chuyển cột `grade` sang kiểu số nguyên** và trả về bảng đã sửa.

**Ví dụ**

- Input: DataFrame `students`

| student_id | name | age | grade |
|---|---|---|---|
| 1 | Ava | 6 | 73.0 |
| 2 | Kate | 15 | 87.0 |

- Output:

| student_id | name | age | grade |
|---|---|---|---|
| 1 | Ava | 6 | 73 |
| 2 | Kate | 15 | 87 |

- Giải thích: giá trị không đổi về mặt toán học (73.0 = 73) nhưng **kiểu dữ liệu** của cột `grade` đổi từ `float64` sang `int64`; trong bảng in ra không còn dấu `.0`.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "student_id": [1, 2],
    "name": ["Ava", "Kate"],
    "age": [6, 15],
    "grade": [73.0, 87.0],
})
print(students)
print()
print(students.dtypes)
print("pandas", pd.__version__)

   student_id  name  age  grade
0           1   Ava    6   73.0
1           2  Kate   15   87.0

student_id      int64
name              str
age             int64
grade         float64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Kiểu dữ liệu (`dtype`) của một cột

Mỗi cột pandas có **một kiểu dữ liệu** chung cho cả cột. Vài kiểu thường gặp:

| dtype | ý nghĩa | ví dụ |
|---|---|---|
| `int64` | số nguyên 64 bit | `73` |
| `float64` | số thực 64 bit (có phần thập phân) | `73.0`, `3.14` |
| `str` / `object` | chuỗi chữ | `"Ava"` |
| `bool` | đúng/sai | `True` |

`df.dtypes` liệt kê kiểu của **mọi cột**, `df["grade"].dtype` cho kiểu của **một cột**.

In [2]:
print(students.dtypes.to_dict())
print(students["grade"].dtype)                 # float64
print(students["age"].dtype)                   # int64
print(students["grade"].tolist())              # [73.0, 87.0]: số thực in kèm .0

{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('float64')}
float64
int64
[73.0, 87.0]


### 3.2. `int(x)` cho từng giá trị

Hàm có sẵn `int(x)` đổi **một số** sang số nguyên **bằng cách cắt bỏ phần thập phân** (về phía 0), **không làm tròn**: `int(73.9)` là `73`, `int(-1.9)` là `-1`. Ta cần hiểu hàm này trước vì `astype(int)` cũng cư xử như vậy.

In [3]:
print(int(73.0))      # 73
print(int(73.9))      # 73  (cắt, KHÔNG làm tròn lên 74)
print(int(-1.9))      # -1  (cắt về phía 0)
print(round(73.9))    # 74  (round mới là làm tròn)

73
73
-1
74


### 3.3. `Series.astype(kiểu)`

`s.astype(int)` trả về **Series mới** có kiểu `int64`. Giá trị thực được **cắt phần thập phân** như `int()`. Phép đổi **không sửa Series cũ**: muốn lưu kết quả phải **gán lại** (`df["grade"] = df["grade"].astype(int)`).

In [4]:
s = students["grade"]
t = s.astype(int)
print(t.tolist(), "| dtype mới:", t.dtype)
print(s.tolist(), "| dtype cũ vẫn:", s.dtype)       # Series gốc không đổi

print(pd.Series([73.9, -1.9, 0.5]).astype(int).tolist())   # [73, -1, 0] -> cắt, không làm tròn

[73, 87] | dtype mới: int64
[73.0, 87.0] | dtype cũ vẫn: float64
[73, -1, 0]


### 3.4. `DataFrame.astype(từ_điển)`

Với **bảng**, `df.astype({"cột": kiểu, ...})` đổi kiểu **chỉ những cột liệt kê** và trả bảng mới. Cột không nhắc tới giữ nguyên kiểu. Cách này đổi nhiều cột trong một lời gọi và không cần gán từng cột.

In [5]:
out = students.astype({"grade": "int64"})
print(out.dtypes.to_dict())
print(students.dtypes.to_dict())        # bảng gốc không đổi

{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}
{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('float64')}


### 3.5. Giá trị thiếu (`NaN`) không đổi được sang `int64`

Kiểu `int64` thường **không có chỗ cho giá trị thiếu**. Nếu cột có `NaN`, `astype(int)` báo lỗi (`IntCastingNaNError`, là một dạng `ValueError`). Hai hướng xử lý: (1) **lấp** giá trị thiếu trước (`fillna`), (2) dùng kiểu **nullable** `"Int64"` (chữ **I** viết hoa), kiểu số nguyên của pandas có thể chứa `<NA>`.

In [6]:
s = pd.Series([73.0, np.nan, 87.0])
try:
    s.astype(int)
except ValueError as e:
    print("ValueError:", type(e).__name__)

print(s.fillna(0).astype(int).tolist())       # lấp 0 rồi mới đổi -> [73, 0, 87]
print(s.astype("Int64").tolist())             # [73, <NA>, 87]: giữ được giá trị thiếu

ValueError: IntCastingNaNError
[73, 0, 87]
[73, <NA>, 87]


### 3.6. `pd.to_numeric` và `downcast`

`pd.to_numeric(s)` biến Series (kể cả chuỗi như `"12"`) thành số. Tham số `downcast="integer"` chọn **kiểu số nguyên nhỏ nhất** đủ chứa dữ liệu (ví dụ `int8`). Kết quả vẫn là số nguyên nhưng **kiểu cụ thể khác `int64`**, nên có thể không khớp bảng kết quả mong đợi nếu máy chấm so sánh kiểu. Dùng khi muốn tiết kiệm bộ nhớ.

In [7]:
g = students["grade"]
print(pd.to_numeric(g, downcast="integer").dtype)    # int8: nhỏ nhất đủ chứa 73, 87
print(pd.to_numeric(pd.Series(["12", "7"])).tolist())  # chuỗi -> số: [12, 7]

int8
[12, 7]


### 3.7. Vì sao tạo `Series` từ danh sách số nguyên Python vẫn ra `int64`?

Khi ta tự duyệt rồi gom `int(x)` vào list `[73, 87]` và gán vào cột, pandas **tự suy ra** kiểu `int64` từ danh sách số nguyên. Ta còn có thể **ép kiểu rõ ràng** bằng `pd.Series(list, dtype="int64")` để chắc chắn (đặc biệt khi danh sách rỗng, lúc đó pandas không có gì để suy luận).

In [8]:
print(pd.Series([73, 87]).dtype)                       # int64
print(pd.Series([], dtype="int64").dtype)              # int64, dù rỗng
print(pd.Series([]).dtype)                             # kiểu khác: pandas không đoán được từ danh sách rỗng

int64
int64
object


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng bảng điểm ghi `73.0`, `87.0`: ta muốn ghi lại là `73`, `87`. Với mỗi ô trong cột `grade`, **bỏ phần `.0`** và **ghi nhớ rằng cột này giờ là số nguyên**.

| nhãn | grade (float) | cắt phần thập phân | grade (int) |
|---|---|---|---|
| 0 | 73.0 | `int(73.0)` | 73 |
| 1 | 87.0 | `int(87.0)` | 87 |

Thuật toán: **áp một phép đổi kiểu cho mọi ô của một cột, rồi ghi lại vào cột**. Vì dữ liệu của bài đều là "số chẵn" (73.0, 87.0), cắt hay làm tròn đều cho cùng kết quả; sự khác biệt chỉ lộ ra với dữ liệu như `73.9` (xem mục 13).

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

**Ý tưởng.** Duyệt từng điểm, gọi `int(...)`, gom vào danh sách; cuối cùng gán **Series kiểu `int64`** vào cột `grade`. Ta ghi rõ `index=df.index` và `dtype="int64"` để mọi trường hợp (kể cả bảng rỗng) đều đúng.

In [9]:
def change_loop(students: pd.DataFrame) -> pd.DataFrame:
    df = students.copy()                         # không đụng bảng gốc
    new_grades = []
    for g in df["grade"]:                        # duyệt từng điểm
        new_grades.append(int(g))                # int() cắt phần thập phân
    df["grade"] = pd.Series(new_grades, index=df.index, dtype="int64")
    return df


res = change_loop(students)
print(res)
print(res.dtypes.to_dict())

   student_id  name  age  grade
0           1   Ava    6     73
1           2  Kate   15     87
{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}


In [10]:
print("grade mới:", res["grade"].tolist(), "| dtype:", res["grade"].dtype)
print("cột khác giữ nguyên kiểu:", res["age"].dtype, res["student_id"].dtype)

grade mới: [73, 87] | dtype: int64
cột khác giữ nguyên kiểu: int64 int64


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng. Vòng lặp `m` lần, mỗi lần `O(1)`: **Time `O(m)`**; danh sách và Series mới `m` phần tử: **Space `O(m)`**.

**Nút thắt.** Vòng lặp Python chậm khi `m` lớn và dài dòng. pandas đổi kiểu cả cột một phát bằng mã C.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác vừa làm bằng tay:

`students["grade"] = students["grade"].astype(int)`

- `astype(int)`: đổi **cả cột** sang `int64` (cắt phần thập phân giống `int()`);
- vế trái: gán đè lên cột `grade`;
- các cột khác không bị động tới.

Các cách khác: **`astype` theo từ điển** trên bảng (Cách 3), **`apply(int)` / `assign`** (Cách 4), **`to_numeric(downcast=...)` hoặc kiểu nullable `Int64`** (Cách 5). Ở đây "chuẩn" là `astype(int)`, vì đề yêu cầu số nguyên thường (`int64`).

## 7. Các cách tiếp theo

### Cách 2 — Gán lại `astype(int)` cho cột (cách chuẩn)

**Giải thích.** `students["grade"].astype(int)` tạo Series mới kiểu `int64`; ta gán lại vào cột. Cách này sửa luôn `students` đầu vào rồi trả về nó; LeetCode chấp nhận.

In [11]:
def change_astype(students: pd.DataFrame) -> pd.DataFrame:
    df = students.copy()
    df["grade"] = df["grade"].astype(int)
    return df


print(change_astype(students).dtypes.to_dict())

{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def changeDatatype(students: pd.DataFrame) -> pd.DataFrame:
    students["grade"] = students["grade"].astype(int)
    return students

**Độ phức tạp Cách 2:** **Time `O(m)`** (một lần quét cột bằng mã C), **Space `O(m)`** (cột `int64` mới).

### Cách 3 — `DataFrame.astype({"grade": "int64"})`

**Giải thích.** Gọi `astype` trên **cả bảng** với từ điển `{tên_cột: kiểu}`; trả về **bảng mới**, không sửa đầu vào. Rất tiện khi đổi nhiều cột cùng lúc (`{"grade": "int64", "age": "float64"}`).

In [13]:
def change_astype_dict(students: pd.DataFrame) -> pd.DataFrame:
    return students.astype({"grade": "int64"})


before = students.copy()
print(change_astype_dict(students).dtypes.to_dict())
pd.testing.assert_frame_equal(students, before)
print("đầu vào không đổi")

{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}
đầu vào không đổi


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def changeDatatype(students: pd.DataFrame) -> pd.DataFrame:
    return students.astype({"grade": int})

**Độ phức tạp Cách 3:** **Time `O(m)`** (chỉ cột được liệt kê bị chuyển), **Space `O(m)`** đến `O(m·c)` tùy cách pandas dùng chung bộ nhớ các cột không đổi.

### Cách 4 — `apply(int)` và `assign`

**Giải thích.** `apply(int)` gọi `int` cho từng ô (chậm hơn `astype` vì gọi hàm Python từng ô). `assign(grade=...)` trả bảng mới (không sửa đầu vào), thường kết hợp với `astype` hoặc `apply` trong chuỗi method.

In [15]:
def change_apply(students: pd.DataFrame) -> pd.DataFrame:
    # apply trên Series rỗng giữ nguyên dtype cũ, nên chốt dtype bằng astype ở cuối
    return students.assign(grade=students["grade"].apply(int).astype("int64"))


def change_assign(students: pd.DataFrame) -> pd.DataFrame:
    return students.assign(grade=lambda d: d["grade"].astype("int64"))


print(change_apply(students).dtypes["grade"], change_assign(students).dtypes["grade"])

int64 int64


**Độ phức tạp Cách 4:** **Time `O(m)`** (`apply` hằng số lớn hơn), **Space `O(m)`**.

### Cách 5 — `to_numeric(downcast="integer")` và kiểu nullable `Int64`

**Giải thích.** Hai cách "gần giống" nhưng **cho kiểu khác `int64`**:

- `pd.to_numeric(..., downcast="integer")` chọn số nguyên **nhỏ nhất** đủ chứa dữ liệu (`int8` cho 73, 87);
- `astype("Int64")` cho số nguyên **nullable** (giữ được `<NA>`).

Giá trị giống nhau, nhưng **dtype khác `int64`**. Ta dùng chúng để tham khảo, còn bài nộp dùng `int64` chuẩn như Cách 2/3.

In [16]:
def change_downcast(students: pd.DataFrame) -> pd.DataFrame:
    return students.assign(grade=pd.to_numeric(students["grade"], downcast="integer"))


def change_nullable(students: pd.DataFrame) -> pd.DataFrame:
    return students.astype({"grade": "Int64"})


a = change_downcast(students)
b = change_nullable(students)
print(a["grade"].dtype, a["grade"].tolist())      # int8
print(b["grade"].dtype, b["grade"].tolist())      # Int64
print("giá trị bằng nhau:", a["grade"].tolist() == b["grade"].tolist() == [73, 87])

int8 [73, 87]
Int64 [73, 87]
giá trị bằng nhau: True


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`** (`int8` tiết kiệm bộ nhớ hơn `int64` tới 8 lần cho cột này).

> Cách 2 và Cách 3 là hai cách nên nộp. Nhớ: `int` ở đây là `int64` trên LeetCode (Linux).

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết, in từng ô; (b) pipeline `dtype trước → astype → gán → dtype sau`.

In [17]:
def trace_loop(students):
    df = students.copy()
    new_grades = []
    for step, (label, old) in enumerate(zip(df.index, df["grade"]), start=1):
        new = int(old)
        print(f"bước {step}: nhãn={label}, grade cũ={old!r} ({type(old).__name__}) -> int() -> {new!r} ({type(new).__name__})")
        new_grades.append(new)
        print(f"          new_grades = {new_grades}")
    df["grade"] = pd.Series(new_grades, index=df.index, dtype="int64")
    print("\ngán vào cột grade -> dtype:", df["grade"].dtype)
    return df


_ = trace_loop(students)

bước 1: nhãn=0, grade cũ=73.0 (float) -> int() -> 73 (int)
          new_grades = [73]
bước 2: nhãn=1, grade cũ=87.0 (float) -> int() -> 87 (int)
          new_grades = [73, 87]

gán vào cột grade -> dtype: int64


In [18]:
def trace_astype(students):
    df = students.copy()
    print("BƯỚC 0 - dtype trước:", df.dtypes.to_dict())
    print(df)

    converted = df["grade"].astype(int)
    print("\nBƯỚC 1 - astype(int) trả Series mới:", converted.tolist(), "| dtype:", converted.dtype)
    print("cột trong bảng lúc này vẫn:", df["grade"].dtype)

    df["grade"] = converted
    print("\nBƯỚC 2 - gán lại vào cột:")
    print(df)
    print("dtype sau:", df.dtypes.to_dict())

    same = df.equals(students.astype({"grade": int}))
    print("\ngiống astype theo từ điển?", same)
    return df


_ = trace_astype(students)

BƯỚC 0 - dtype trước: {'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('float64')}
   student_id  name  age  grade
0           1   Ava    6   73.0
1           2  Kate   15   87.0

BƯỚC 1 - astype(int) trả Series mới: [73, 87] | dtype: int64
cột trong bảng lúc này vẫn: float64

BƯỚC 2 - gán lại vào cột:
   student_id  name  age  grade
0           1   Ava    6     73
1           2  Kate   15     87
dtype sau: {'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}

giống astype theo từ điển? True


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | `grade` | `[73.0, 87.0]`, `float64` |
| 1 | `grade.astype(int)` | Series `[73, 87]`, `int64` (bảng chưa đổi) |
| 2 | gán vào cột | bảng có `grade` kiểu `int64`, các cột khác không đổi |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def changeDatatype(students: pd.DataFrame) -> pd.DataFrame:
    students["grade"] = students["grade"].astype(int)
    return students


demo = students.copy()           # dùng bản sao để các cell sau còn dữ liệu gốc
print(changeDatatype(demo))
print(demo.dtypes.to_dict())

   student_id  name  age  grade
0           1   Ava    6     73
1           2  Kate   15     87
{'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'grade': dtype('int64')}


## 10. Vì sao thuật toán đúng

- **Đúng kiểu đích:** `astype(int)` biến mọi phần tử của cột thành `int64`; đề yêu cầu cột `grade` là số nguyên.
- **Đúng giá trị:** với điểm vốn là số nguyên lưu dạng thực (73.0, 87.0), phần thập phân bằng 0 nên cắt không làm mất thông tin: `int(73.0) == 73`.
- **Đúng vị trí:** phép gán `df["grade"] = Series` ghép theo nhãn index; Series được tính từ chính cột đó nên mỗi giá trị mới nằm đúng hàng cũ.
- **Cột khác không đổi:** ta chỉ gán vào cột `grade`; `student_id`, `name`, `age` giữ nguyên giá trị và kiểu.
- **Tính chất kiểm được:** kết quả `grade` có `dtype == int64` và `result["grade"] == student["grade"]` (so sánh giá trị số) với mọi hàng khi dữ liệu gốc không có phần thập phân.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

- **Time:** `O(m)`: mỗi ô của cột `grade` được xử lý đúng một lần. `astype` chạy bằng mã C nên hằng số nhỏ; vòng lặp Python / `apply` có hằng số lớn hơn nhưng cùng bậc.
- **Space:** `O(m)` cho cột mới (cột `float64` cũ bị thay, nên bộ nhớ ổn định); `O(m·c)` nếu tạo bảng mới bằng `astype` trên bảng/`assign`.
- Không thể nhanh hơn `O(m)` vì mọi ô phải được đọc ít nhất một lần.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`; mỗi lần gọi truyền **bản sao** của bảng để các hàm sửa tại chỗ (Cách 2) không làm hỏng test sau. Cách 5 (`downcast`, `Int64`) cho kiểu khác `int64` nên ta kiểm tra riêng ở dưới.

In [20]:
impls = {
    "loop": change_loop,
    "astype": change_astype,
    "astype_dict": change_astype_dict,
    "apply": change_apply,
    "assign": change_assign,
    "submit": changeDatatype,
}

expected_example = pd.DataFrame({
    "student_id": [1, 2],
    "name": ["Ava", "Kate"],
    "age": [6, 15],
    "grade": [73, 87],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(students.copy()), expected_example)
print("PASS ví dụ của đề (mọi cách), dtype grade =", expected_example["grade"].dtype)

PASS ví dụ của đề (mọi cách), dtype grade = int64


### 12.1. Các trường hợp biên

In [21]:
def make(grades):
    n = len(grades)
    return pd.DataFrame({
        "student_id": list(range(1, n + 1)),
        "name": pd.Series([f"s{i}" for i in range(n)], dtype="str"),
        "age": pd.Series([10] * n, dtype="int64"),
        "grade": pd.Series(grades, dtype="float64"),
    })


def check_all(df, expected_grades, label):
    for key, fn in impls.items():
        out = fn(df.copy())
        assert out["grade"].tolist() == expected_grades, (label, key, out["grade"].tolist())
        assert out["grade"].dtype == "int64", (label, key, out["grade"].dtype)
    print("PASS", label, "->", expected_grades)


check_all(make([73.0]), [73], "một hàng")
check_all(make([0.0, 100.0, 50.0]), [0, 100, 50], "biên 0 và 100")
check_all(make([-5.0, 5.0]), [-5, 5], "số âm")
check_all(make([2.0**40]), [2**40], "số lớn (vẫn nằm trong int64)")
check_all(make([60.0, 60.0, 60.0]), [60, 60, 60], "điểm bằng nhau")

PASS một hàng -> [73]
PASS biên 0 và 100 -> [0, 100, 50]
PASS số âm -> [-5, 5]
PASS số lớn (vẫn nằm trong int64) -> [1099511627776]
PASS điểm bằng nhau -> [60, 60, 60]


### 12.2. Bảng rỗng, phần thập phân, giá trị thiếu, đầu vào, và phản ví dụ

In [22]:
# Bảng rỗng: kiểu vẫn phải là int64
empty = make([])
for key, fn in impls.items():
    out = fn(empty.copy())
    assert len(out) == 0 and out["grade"].dtype == "int64", key
print("PASS bảng rỗng: dtype =", changeDatatype(empty.copy())["grade"].dtype)

# Phần thập phân bị cắt (không làm tròn)
frac = make([73.9, 87.1, -1.9])
assert changeDatatype(frac.copy())["grade"].tolist() == [73, 87, -1]
print("PASS phần thập phân bị cắt:", changeDatatype(frac.copy())["grade"].tolist())

# Giá trị thiếu: astype(int) báo lỗi, Int64 thì giữ được
miss = make([73.0, np.nan])
try:
    changeDatatype(miss.copy())
    raise AssertionError("đáng lẽ phải lỗi")
except ValueError:
    print("PASS NaN: astype(int) báo ValueError")
assert change_nullable(miss)["grade"].isna().tolist() == [False, True]
print("PASS NaN: Int64 giữ được <NA>")

# Cách 5 cho giá trị giống nhau nhưng dtype khác int64
assert change_downcast(students)["grade"].tolist() == [73, 87] and str(change_downcast(students)["grade"].dtype) == "int8"
assert change_nullable(students)["grade"].tolist() == [73, 87] and str(change_nullable(students)["grade"].dtype) == "Int64"
print("PASS downcast / Int64: giá trị đúng, dtype khác int64")

# Các cách "thuần" không sửa đầu vào, các cột khác không đổi kiểu
before = students.copy()
for key in ["loop", "astype_dict", "apply", "assign"]:
    impls[key](students)
pd.testing.assert_frame_equal(students, before)
out = changeDatatype(students.copy())
assert out["age"].dtype == students["age"].dtype and out["name"].tolist() == students["name"].tolist()
print("PASS đầu vào không đổi; cột khác giữ kiểu")

# Phản ví dụ: không gán lại thì cột vẫn float
tmp = students.copy()
tmp["grade"].astype(int)
assert tmp["grade"].dtype == "float64"
print("PASS phản ví dụ: astype không gán lại -> cột vẫn float64")

PASS bảng rỗng: dtype = int64
PASS phần thập phân bị cắt: [73, 87, -1]
PASS NaN: astype(int) báo ValueError
PASS NaN: Int64 giữ được <NA>
PASS downcast / Int64: giá trị đúng, dtype khác int64
PASS đầu vào không đổi; cột khác giữ kiểu
PASS phản ví dụ: astype không gán lại -> cột vẫn float64


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
import random

random.seed(2886)
for trial in range(200):
    n = random.randint(0, 20)
    grades = [float(random.randint(0, 100)) for _ in range(n)]
    df = make(grades)
    ref = df.assign(grade=pd.Series([int(g) for g in grades], dtype="int64"))
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df.copy()), ref)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Gọi `astype` nhưng quên gán lại

In [24]:
def sai_quen_gan(students):
    students["grade"].astype(int)         # tạo Series mới rồi vứt đi
    return students


print("sai  :", sai_quen_gan(students.copy())["grade"].dtype)              # float64
print("đúng :", changeDatatype(students.copy())["grade"].dtype)             # int64

sai  : float64
đúng : int64


**Vì sao sai:** `astype` **trả về** Series mới, không sửa cột cũ. Phải gán lại: `students["grade"] = students["grade"].astype(int)`.

### Lỗi 2: Nhầm "cắt" với "làm tròn"

In [25]:
g = pd.Series([73.9, 59.5, 88.1])
print("astype(int)         :", g.astype(int).tolist())          # [73, 59, 88]  cắt phần thập phân
print("round().astype(int) :", g.round().astype(int).tolist())  # [74, 60, 88]  làm tròn rồi đổi kiểu

astype(int)         : [73, 59, 88]
round().astype(int) : [74, 60, 88]


**Vì sao dễ nhầm:** `astype(int)` **cắt** về phía 0, không làm tròn (`73.9 -> 73`). Nếu đề yêu cầu làm tròn, phải `round()` trước. Với dữ liệu của bài này (toàn `.0`) hai cách giống nhau, nhưng bài khác có thể khác.

### Lỗi 3: Đổi cột có giá trị thiếu sang `int`

In [26]:
g = pd.Series([73.0, np.nan, 87.0])
try:
    g.astype(int)
except ValueError as e:
    print("ValueError:", str(e)[:60], "...")
print("lấp trước :", g.fillna(0).astype(int).tolist())
print("hoặc Int64:", g.astype("Int64").tolist())

ValueError: Cannot convert non-finite values (NA or inf) to integer.Repl ...
lấp trước : [73, 0, 87]
hoặc Int64: [73, <NA>, 87]


**Vì sao sai:** `int64` của NumPy không biểu diễn được `NaN`. Hãy lấp giá trị thiếu bằng `fillna(...)` (nếu hợp lý với đề), hoặc dùng kiểu nullable `"Int64"`.

### Lỗi 4: Chuyển chuỗi không phải số sang số

In [27]:
s = pd.Series(["12", "7", "abc"])
try:
    s.astype(int)
except ValueError as e:
    print("ValueError:", e)
print("an toàn hơn:", pd.to_numeric(s, errors="coerce").tolist())     # chuỗi lỗi -> NaN

ValueError: invalid literal for int() with base 10: 'abc'
an toàn hơn: [12.0, 7.0, nan]


**Vì sao sai:** chuỗi `"abc"` không phải số nên `astype(int)` báo lỗi. `pd.to_numeric(..., errors="coerce")` biến giá trị không đổi được thành `NaN` để bạn xử lý tiếp. Bài này dữ liệu đã là số thực nên không gặp, nhưng nên biết khi gặp dữ liệu bẩn.

### Lỗi 5: Trả về Series thay vì cả bảng

In [28]:
def sai_return_series(students):
    return students["grade"].astype(int)


r = sai_return_series(students.copy())
print("sai  : kiểu trả về =", type(r).__name__)
print("đúng : kiểu trả về =", type(changeDatatype(students.copy())).__name__)

sai  : kiểu trả về = Series
đúng : kiểu trả về = DataFrame


**Vì sao sai:** đề yêu cầu trả **cả bảng** (`student_id`, `name`, `age`, `grade`), tức `DataFrame`. Trả Series làm mất các cột khác.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đổi kiểu dữ liệu / chuyển sang số nguyên / ép kiểu" → nghĩ tới **`astype`**.
- Công thức nhớ: **`df["col"] = df["col"].astype(kiểu)`**; nhiều cột: **`df.astype({"a": int, "b": float})`**.
- Luôn **kiểm tra `df.dtypes`** trước và sau.
- Có `NaN` → lấp (`fillna`) trước hoặc dùng `"Int64"` (chữ **I** hoa).
- Dữ liệu chuỗi bẩn → `pd.to_numeric(..., errors="coerce")`.
- Cần làm tròn → `round()` trước `astype(int)`. Đổi sang thời gian → `pd.to_datetime`; sang chuỗi → `astype(str)`.
- Muốn tiết kiệm bộ nhớ → `downcast` hoặc kiểu nhỏ (`int32`, `float32`) nhưng cẩn thận tràn số.

## 15. Biến thể của bài

1. Đổi **nhiều cột** cùng lúc: `df.astype({"grade": "int64", "age": "float64"})`.
2. Đổi sang **chuỗi**: `df["grade"].astype(str)`.
3. Đổi sang kiểu **danh mục** `category` (tiết kiệm bộ nhớ khi giá trị lặp nhiều): `df["name"].astype("category")`.
4. Đổi cột chuỗi `"2024-05-01"` sang **ngày**: `pd.to_datetime(df["date"])`.
5. Làm tròn rồi đổi kiểu: `df["grade"].round().astype(int)`.
6. Đổi **mọi cột số thực** sang số nguyên: `df.select_dtypes("float64").astype(int)`.

Ta thử vài biến thể:

In [29]:
v = students.copy()
print("1. nhiều cột:", v.astype({"grade": "int64", "age": "float64"}).dtypes.to_dict())
print("2. chuỗi    :", v["grade"].astype(str).tolist())
print("3. category :", v["name"].astype("category").dtype)
print("4. ngày     :", pd.to_datetime(pd.Series(["2024-05-01"])).dtype)
print("6. mọi cột float:", v.select_dtypes("float64").astype(int).dtypes.to_dict())

1. nhiều cột: {'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('float64'), 'grade': dtype('int64')}
2. chuỗi    : ['73.0', '87.0']
3. category : category
4. ngày     : datetime64[us]
6. mọi cột float: {'grade': dtype('int64')}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `int()` | tự đổi từng ô | `O(m)` (Python, chậm) | `O(m)` | hiểu bản chất; không dùng trên dữ liệu lớn |
| 2. `df["col"] = df["col"].astype(int)` | đổi cả cột + gán | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `df.astype({"col": int})` | đổi theo từ điển, bảng mới | `O(m)` | `O(m)` đến `O(m·c)` | đổi nhiều cột, không sửa đầu vào |
| 4. `apply(int)` / `assign` | gọi hàm từng ô / bảng mới | `O(m)` (hằng số lớn) | `O(m)` | chuỗi method, phép đổi tùy biến |
| 5. `to_numeric(downcast)` / `Int64` | số nguyên nhỏ nhất / nullable | `O(m)` | `O(m)` | tiết kiệm bộ nhớ / có giá trị thiếu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `int(7.9)`, `int(-7.9)`, `pd.Series([1.5, 2.5]).astype(int).tolist()`, `students["grade"].dtype`, `students["age"].astype(float).tolist()`.

**Bài 2 (tính tay).** Với `grade = [59.5, 60.4, 99.9, 0.0]`: viết kết quả của `astype(int)`; viết kết quả của `round().astype(int)` (lưu ý `round` của pandas làm tròn về số chẵn gần nhất khi đúng `.5`: `59.5 -> 60`, `2.5 -> 2`).

**Bài 3 (điền TODO).** Hoàn thành `change_todo(df)` (không sửa bảng đầu vào).

In [30]:
def change_todo(df):
    # TODO 1: out = df.copy()
    # TODO 2: out["grade"] = out["grade"].astype(int)
    # TODO 3: return out
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `change_loop2(students)` bằng vòng lặp thuần Python (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `change_rounded(df)`: đổi `grade` sang `int64` nhưng **làm tròn** thay vì cắt. (b) Viết `change_safe(df)`: nếu `grade` có `NaN` thì lấp bằng `0` trước, rồi đổi sang `int64`; nếu không có `NaN` thì chỉ đổi kiểu.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7`, `-7`, `[1, 2]`, `float64`, `[6.0, 15.0]`. Bài 2: `astype(int)` bỏ phần sau dấu chấm.
- **Mức 2 (vừa):** Bài 2: `astype(int)` -> `[59, 60, 99, 0]`; `round().astype(int)` -> `[60, 60, 100, 0]`. Bài 3: ba dòng đã ghi trong comment. Bài 4: tạo list, duyệt từng giá trị, `int(g)`, gán `pd.Series(list, index=df.index, dtype="int64")`.
- **Mức 3 (sâu):** Bài 5(a): `df.assign(grade=df["grade"].round().astype("int64"))`. Bài 5(b): `df.assign(grade=df["grade"].fillna(0).astype("int64"))` (việc `fillna` khi không có `NaN` không làm thay đổi gì, nên dùng luôn được).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert int(7.9) == 7 and int(-7.9) == -7
assert pd.Series([1.5, 2.5]).astype(int).tolist() == [1, 2]
assert str(students["grade"].dtype) == "float64"
assert students["age"].astype(float).tolist() == [6.0, 15.0]

# Bài 2
g2 = pd.Series([59.5, 60.4, 99.9, 0.0])
assert g2.astype(int).tolist() == [59, 60, 99, 0]
assert g2.round().astype(int).tolist() == [60, 60, 100, 0]

# Bài 3
def change_todo_ans(df):
    out = df.copy()
    out["grade"] = out["grade"].astype(int)
    return out

before = students.copy()
pd.testing.assert_frame_equal(change_todo_ans(students), expected_example)
pd.testing.assert_frame_equal(students, before)      # bảng đầu vào không đổi

# Bài 4
def change_loop2(students):
    df = students.copy()
    new_grades = []
    for g in df["grade"]:
        new_grades.append(int(g))
    df["grade"] = pd.Series(new_grades, index=df.index, dtype="int64")
    return df

pd.testing.assert_frame_equal(change_loop2(students), expected_example)

# Bài 5(a)
def change_rounded(df):
    return df.assign(grade=df["grade"].round().astype("int64"))

assert change_rounded(make([73.9, 87.1, 59.4]))["grade"].tolist() == [74, 87, 59]

# Bài 5(b)
def change_safe(df):
    return df.assign(grade=df["grade"].fillna(0).astype("int64"))

assert change_safe(make([73.0, np.nan]))["grade"].tolist() == [73, 0]
assert change_safe(students)["grade"].tolist() == [73, 87]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi kiểu một cột)

```python
import pandas as pd

def changeDatatype(students: pd.DataFrame) -> pd.DataFrame:
    students["grade"] = students["grade"].astype(int)
    return students
```

**Checklist**

- [ ] Có **phép gán** `students["grade"] = ...` (hoặc dùng `astype({...})` và `return` kết quả)?
- [ ] Đổi đúng cột `grade`, kiểu đích là số nguyên `int64` (không phải `int8`, không phải `Int64` nếu đề không yêu cầu)?
- [ ] Biết `astype(int)` **cắt** phần thập phân; cần làm tròn thì `round()` trước?
- [ ] Cột có `NaN` thì đã xử lý (`fillna` hoặc `Int64`)?
- [ ] **Trả về** DataFrame (không trả Series)?
- [ ] Đã kiểm tra `df.dtypes` sau khi đổi, cột khác không đổi kiểu?
- [ ] Skeleton đúng `def changeDatatype(students: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?